In [ ]:
import torch
import torch_geometric
from torch_geometric.data import Dataset, Data
from torch_geometric.data import HeteroData
import numpy as np 
import os
import math
from tqdm import tqdm
import yaml
import time
import os
import sys
import pandas as pd
import cv2
import pickle
import gc
from matplotlib import pyplot as plt
from collections import Counter
import numpy as np
import scipy.sparse as sp
import torch
import torchvision
import torch.nn as nn
from torch.nn import Linear
import torch.nn.functional as F
from torch.nn.functional import normalize
from scipy.spatial.distance import cdist
from scipy.spatial.distance import euclidean
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics.pairwise import euclidean_distances
import torch_geometric
from torch_geometric.data import Dataset, Data
from torch_geometric.utils.convert import to_networkx
import networkx as nx
from sklearn.preprocessing import LabelEncoder
import torchvision.ops.boxes as bops
import itertools
from torch_geometric.nn import SplineConv

In [ ]:
def modify_annotation_data(input_video_path, annotation_list):
    """
    -TO DO: Convert the time information of actions from seconds to frames

    -Inputs/ Arguments:
    annotation_list: df with startTime & endTime into seconds
    fps: frame per seconds at which the input video was recorded

    -Outputs/ Returns:
    annotation_list: dataframe containing both second and frame information
    """
    new_df = pd.DataFrame(columns=["task", "start_frame", "end_frame"])
    cap = cv2.VideoCapture(input_video_path)
    fps_video = cap.get(cv2.CAP_PROP_FPS)

    for i in range(len(annotation_list)):
        task_i = annotation_list['task'].iloc[i]
        start_i = pd.Series(pd.to_timedelta([annotation_list["startTime"].iloc[i]])).dt.total_seconds()
        start_i = start_i[0] * fps_video
        end_i = pd.Series(pd.to_timedelta([annotation_list["endTime"].iloc[i]])).dt.total_seconds()
        end_i = end_i[0] * fps_video
        new_df.loc[i] = [task_i, start_i, end_i]
    annotation_list["start_frame"] = new_df["start_frame"].to_numpy()
    annotation_list["end_frame"] = new_df["end_frame"].to_numpy()
    return annotation_list

In [ ]:
def GetShiftingWindows(thelist, size, overlap):
    windows = []
    i = 0
    while i < len(thelist):
        if i + size <= len(thelist):
            windows.append(thelist[i:i+size])
        else:
            windows.append(thelist[i:])
            break
        i += size - overlap
    return windows

In [ ]:
def load_labels(combined_y, frame_window):
    last_frame = frame_window[-1]
    ind = 0
    for start, end in zip(combined_y["start_frame"], combined_y["end_frame"]):
        if start <= last_frame <= end:
            break
        ind += 1

    task = combined_y["task"][ind]
    # print(frame_window,ind, task)
    # Create a list of dictionaries
    data = [{"FrameWindow": frame_window, "Task": task}]
    # Create a dataframe
    df = pd.DataFrame(data)
    # print(df)

    return df, task

In [ ]:
def encode_onehot(all_tasks,frame_window_tasks):
    lb = LabelBinarizer()
    lb.fit(all_tasks["task"])
    classesID = list(lb.classes_)
    num_classes = len(classesID)
        # Create a text file with the ordered classes so that we can decode it afterwards
    with open(
            "classes.txt",'w'
    ) as output:
        for row in classesID:
            output.write(str(row) + "\n")
    y = lb.transform(frame_window_tasks)
    return y

In [ ]:
def pickle_data(data, folder, filename):
    with open(os.path.join(folder, filename), 'wb') as handle:
        pickle.dump(data, handle)
        
def unpickle_data(folder, filename):
    with open(os.path.join(folder, filename), 'rb') as handle:
        data = pickle.load(handle)
        return data

In [ ]:
def diag_block_mat(tList):
    blkXsize = [ tbl.shape[1] for tbl in tList ]
    outBlocks = []
    for i, tbl in enumerate(tList):
        tBefore = np.zeros((tbl.shape[0], sum(blkXsize[:i])))
        tAfter = np.zeros((tbl.shape[0], sum(blkXsize[i+1:])))
        outBlocks.append(np.hstack([tBefore, tbl, tAfter]))
    return np.vstack(outBlocks)

In [ ]:
def bb_intersection_over_union(boxA, boxB):
    # determine the (x, y)-coordinates of the intersection rectangle
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    # compute the area of intersection rectangle
    interArea = max(0, xB - xA + 1) * max(0, yB - yA + 1)
    # compute the area of both the prediction and ground-truth
    # rectangles
    boxAArea = (boxA[2] - boxA[0] + 1) * (boxA[3] - boxA[1] + 1)
    boxBArea = (boxB[2] - boxB[0] + 1) * (boxB[3] - boxB[1] + 1)
    # compute the intersection over union by taking the intersection
    # area and dividing it by the sum of prediction + ground-truth
    # areas - the interesection area
    iou = interArea / float(boxAArea + boxBArea - interArea)
    # return the intersection over union value
    return iou

In [ ]:
def construct_iou_adjacency_matrix(frame_window_classes, frame_window_boxes, total_object_count):
    # Initialize the adjacency matrix for the entire frame window
    iou_matrix_list = []
    class_indices = []
    total_classes = 0
    frame_indices = []

    # Iterate over each frame in the frame window #FW
    for frame_idx, (frame_classes, frame_boxes) in enumerate(zip(frame_window_classes, frame_window_boxes)):
        # Initialize the IOU matrix for the current frame
        iou_matrix = np.zeros((len(frame_boxes), len(frame_boxes)))
        for object_idx, frame_class in enumerate(frame_classes):
            continuous_object_idx = total_classes + object_idx
            class_indices.append((continuous_object_idx,frame_idx,frame_class,frame_boxes[object_idx]))
        total_classes += len(frame_classes)

        # Calculate IOU and update the IOU matrix for each pair of objects in the same frame #Frame
        for i, box1 in enumerate(frame_boxes):
            for j, box2 in enumerate(frame_boxes):
                if i == j: #skip self loop conections
                    continue
                iou = bb_intersection_over_union(box1, box2)

                fc1 = frame_classes[i]
                fc2 = frame_classes[j]

                # Update the IOU matrix
                iou_matrix[i, j] = iou
                iou_matrix[j, i] = iou
        iou_matrix_list.append(iou_matrix)
            
    # Accumulate the IOU matrix of the current frame into the adjacency matrix
       
    iou_intra_matrix = diag_block_mat(iou_matrix_list)

    iou_inter_matrix = np.zeros((total_object_count, total_object_count))
    
    for i, (object_idx_i,frame_idx_i,frame_class_i, frame_box_i) in enumerate(class_indices):  
        frame_class_counts = {}        
        for j, (object_idx_j,frame_idx_j, frame_class_j, frame_box_j) in enumerate(class_indices):
                       
            if i == j: #skip self loop conections
                continue
            if frame_idx_i == frame_idx_j: #skip same frame connections
                continue 
            if frame_class_i != frame_class_j:
                continue
            elif frame_class_i == 12 and frame_class_j != 12:
                continue
            elif frame_class_i != 12 and frame_class_j == 12:
                continue
            elif frame_class_i !=12 and frame_class_j != 12:
                continue
            else:
                iou = bb_intersection_over_union(frame_box_i, frame_box_j)
                iou_inter_matrix[i, j] = iou
                iou_inter_matrix[j, i] = iou
            
            if frame_class_j not in frame_class_counts:
                frame_class_counts[frame_class_j] = {}
            if frame_idx_j not in frame_class_counts[frame_class_j]:
                frame_class_counts[frame_class_j][frame_idx_j] = {'count': 1, 'object_ids': [object_idx_j],'bounding_boxes': [frame_box_j]}
            else:
                frame_class_counts[frame_class_j][frame_idx_j]['count'] += 1
                frame_class_counts[frame_class_j][frame_idx_j]['object_ids'].append(object_idx_j)
                frame_class_counts[frame_class_j][frame_idx_j]['bounding_boxes'].append(frame_box_j)
                
#     adj_matrix = iou_intra_matrix + iou_inter_matrix
    iou_inter_matrix = torch.tensor(iou_inter_matrix, dtype=torch.float32)
    iou_intra_matrix = torch.tensor(iou_intra_matrix, dtype=torch.float32)

#     adj_matrix = torch.tensor(adj_matrix, dtype=torch.float32)
#     adj_matrix = adj_matrix + adj_matrix.T.multiply(adj_matrix.T > adj_matrix) - adj_matrix.multiply(adj_matrix.T > adj_matrix)
#     row_sums = adj_matrix.sum(dim=1, keepdim=True)
#     normalized_matrix = adj_matrix / row_sums
    return iou_intra_matrix, iou_inter_matrix


In [ ]:
with open("data.yml", "r") as file_handle:
    yaml_file_params = yaml.load(file_handle, Loader=yaml.Loader)

# Assembly operation selected
assembly = 'L10'

# Identify the assembly operation data
assembly_operation = yaml_file_params[assembly]

# Training data location
training_videos_dir = os.path.join(os.path.dirname(os.getcwd()), *assembly_operation["data_dir"]["training"])
training_video_names = [os.path.join(training_videos_dir, cycle) for cycle in os.listdir(training_videos_dir) if
                        cycle.split(".")[-1] == "mp4"]
training_annotation_names = ["human_" + cycle.split("/")[-1][:-4] + ".csv" for cycle in training_video_names]

# Assert that all the training data is labelled
assert len(training_video_names) == len(training_annotation_names), "The labelling and the data for training " \
                                                                    "does not match"
# Get the full path of the training data
training_videos_fullpath = [os.path.join(training_videos_dir, cycle_name) for cycle_name in
                            training_video_names]

training_annotations_fullpath = [os.path.join(training_videos_dir, annotation_name) for annotation_name in
                                 training_annotation_names]


RESULTS_DIR = "/data1/GCNN/RPN_Train_Results"
window_size = 30
overlap = 15


In [ ]:
# Make a final assertion to ensure they match
start_time = time.time()
data_list = []
all_tasks = pd.DataFrame()
for video_name, annotation_name in zip(training_videos_fullpath, training_annotations_fullpath):
    aug_video_name = video_name.split(os.path.sep)[-1].split(".")[0]
    aug_annotation_name = ("_".join(annotation_name.split(os.path.sep)[-1].split("_")[1:])).split(".")[0]
    assert aug_video_name == aug_annotation_name, "The annotations and the cycle do not match. Please check " \
                                                  "training directory"
# Load into DataFrame
y_list = []
# Initialize frame_data_mapping dictionary
for annotation_path in training_annotations_fullpath:
    y_list.append(pd.read_csv(annotation_path, header=0, names=["task", "startTime", "endTime"]))
# Process the DataFrame
# y = []
video_counter = 0
# all_frame_window_tasks = []  # Initialize the frame_window_data list
total_frame_window_count = 0  # Initialize the counter for total frame windows

for video, annotation in zip(training_videos_fullpath, y_list):
    frame_data_mapping = {}
    y = []
    combined_y = None
#     if 'WIN_20220128_14_20_20_Pro_Jan28_cycle1' not in video:
#         continue 

    processed_y = modify_annotation_data(input_video_path=video, annotation_list=annotation)
    y.append(processed_y)
    # Stack the y together
    imp_cols = ["video", "task", "startTime", "endTime", "start_frame", "end_frame"]
    for index, df_y in enumerate(y):
        # Set the video ID
        df_y["video"] = video_counter

        if index == 0:
            combined_y = df_y[imp_cols]
            index += 1
        else:
            combined_y = pd.concat([combined_y, df_y[imp_cols]], axis=0, ignore_index=True)
    all_tasks = pd.concat([all_tasks, combined_y])
    sys.stdout.write(f"Total number of task instances: {combined_y['task'].count()}")
    # print("\n")
    video_counter += 1
    combined_y["start_frame"] = combined_y["start_frame"].astype(int)
    combined_y["end_frame"] = combined_y["end_frame"].astype(int)
    print("video:", video)
#     print(combined_y)

    for index, task in combined_y.iterrows():
        seq = list(range(int(task['start_frame']), int(task['end_frame']) + 1))
        frame_window_list = GetShiftingWindows(seq, window_size, overlap)

        for frame_window in frame_window_list:
            frame_metadata_list = []  # To store the frame metadata for each frame in the frame window
            labels, tasks = load_labels(combined_y, frame_window)
            adjusted_frame_window = []  # List to store adjusted frame window

            for frames in frame_window:
                frame_path = os.path.join(RESULTS_DIR, video.split('/')[-1][:-4], str(frames) + '.pickle')
                if os.path.exists(frame_path):
                    with open(frame_path, 'rb') as handle:
                        frame_metadata = pickle.load(handle)
                        frame_metadata['features'] = frame_metadata['features'].cpu()
                        frame_metadata_list.append(frame_metadata) #store corresponding metadata for each frame window
                        adjusted_frame_window.append(frames)  # Add the frame to the adjusted frame window
            if len(adjusted_frame_window) == 0:
                continue
            frame_data_mapping[tuple(adjusted_frame_window)] = {"metadata": frame_metadata_list, "task": tasks}

            # Associate frame_window with frame_metadata list and task labels

    # Number of frame_windows
    print("Number of frame_windows per video:", len(frame_data_mapping))
    torch.cuda.empty_cache()
    
    for index,(frame_window, frame_window_data) in enumerate(frame_data_mapping.items()):# inside all frame windows in a specific video
#         print(frame_window)
        total_object_count =0
        #total_object_count_classes =0
        original_frame_window_features = [] # Initialize the list to store original features for each frame window in the current video
        frame_window_features = [] # Initialize the list to store modified features for each frame window in the current video
        frame_window_classes =[] # Initialize the list to store classes for each frame window
        frame_window_boxes =[] # Initialize the list to store boxes for each frame window
        frame_window_tasks =[]
        frame_window_tasks.append(frame_window_data['task'])
#         print(frame_window_tasks)
        total_frame_window_count += 1
        frame_window_metadata_list = frame_window_data["metadata"] #all frame metadata for all the frames within a specific frame window
        # inside all frames within a specific frame window within a specific video:
        for frame_window_idx, frame_window_metadata in enumerate(frame_window_metadata_list):
            frame_features = frame_window_metadata["features"] #features of all the frames within a specific frame window
            frame_boxes = frame_window_metadata["boxes"] #bounding box coordinates of all the frames within a specific frame window
            frame_classes = frame_window_metadata["class"] #classes of all the frames within a specific frame window
            
            total_object_count+=len(frame_boxes)
            frame_window_classes.append(frame_classes)
#         print(frame_window_classes)
            frame_window_boxes.append(frame_boxes)
            original_frame_window_features.append(frame_features)

#         for boxes, classes in enumerate(zip(frame_window_boxes, frame_window_classes)):
#             print(boxes)
#             print(classes)
# #             print(features.shape)
#         for features in original_frame_window_features:
#             print(features.shape)
# #             print(features.shape)
       
                
            for object_idx, object_feature in enumerate(frame_features):# inside all objects within a specific frame
                max_pool = nn.MaxPool2d(kernel_size=(7, 7))
                pooled_features_per_object = max_pool(object_feature)
                averaged_features_per_object = torch.mean(pooled_features_per_object, dim=(-2, -1))
                frame_window_features.append(averaged_features_per_object)
                
        frame_window_features = torch.stack(frame_window_features, dim=0)
        row_sums = frame_window_features.sum(dim=1, keepdim=True)
        frame_window_features = frame_window_features / row_sums
        labels = encode_onehot(all_tasks,frame_window_tasks)
#         print(labels)
        labels = torch.LongTensor(np.where(labels)[1])
#         print(labels)
#         inverse_dists = calculate_inverse_distances(frame_window_features)
        adj_spatial, adj_temporal = construct_iou_adjacency_matrix(frame_window_classes,frame_window_boxes,
                                             total_object_count)
        edge_index_spatial = adj_spatial.nonzero().t()
        edge_index_spatial = edge_index_spatial.to(torch.long)      
        edge_weight_spatial = adj_spatial[edge_index_spatial[0], edge_index_spatial[1]]
        edge_weight_spatial = edge_weight_spatial.to(torch.float)

        edge_index_temporal = adj_temporal.nonzero().t()
        edge_index_temporal = edge_index_temporal.to(torch.long)
        edge_weight_temporal = adj_temporal[edge_index_temporal[0], edge_index_temporal[1]]
        edge_weight_temporal = edge_weight_temporal.to(torch.float)
#         print("Edge Index:")
#         print(edge_index)
#         print("Edge Weight:")
#         print(edge_weight)
#         print(edge_weight.shape)
        data = HeteroData(frame_window={'x': frame_window_features})
        data['frame_window'].y = labels
#         data['frame_window'].x = frame_window_features
        data['frame_window','spatial','frame_window'].edge_index = edge_index_spatial
        data['frame_window','spatial','frame_window'].edge_weight = edge_weight_spatial
        data['frame_window','temporal','frame_window'].edge_index = edge_index_temporal        
        data['frame_window','temporal','frame_window'].edge_weight = edge_weight_temporal
        
        
#         data = Data(frame_window_features,edge_index = edge_index,edge_weight = edge_weight, y =labels)
        data_list.append(data)
#         print(data)
#         print(f'Number of nodes: {data.num_nodes}')
#         print(f'Number of edges: {data.num_edges}')
#         print(f'Has isolated nodes: {data.has_isolated_nodes()}')
#         print(f'Has self-loops: {data.has_self_loops()}')
#         print(f'Is undirected: {data.is_undirected()}')
#         print(f'Label: {data.y}')

num_data_objects = len(data_list)
print(f"Number of Data objects in the list: {num_data_objects}")

print("--- %s seconds ---" % (time.time() - start_time))
# print(all_tasks)

In [ ]:
# for data in data_list:
#     node_types, edge_types = data.metadata()
#     print(node_types)
#     print(edge_types)
#     print(data)
#     print('features',data.x.device) 
#     print(data.device)

In [ ]:
# from torch_geometric.utils import is_undirected
# for data in data_list:
#     num_nodes = data['frame_window'].num_nodes
#     print(data['frame_window','spatial','frame_window'].edge_index)
#     print(data['frame_window','temporal','frame_window'].edge_index)
#     print(num_nodes)

In [ ]:

# for data in data_list:
#     print(f'Has isolated nodes: {data.has_isolated_nodes()}')


In [ ]:
# from torch_geometric.utils import contains_self_loops

# for data in data_list:
#     frame_window_edge_index_spatial = data['frame_window', 'spatial', 'frame_window'].edge_index
#     frame_window_edge_index_temporal = data['frame_window', 'temporal', 'frame_window'].edge_index
    
#     has_self_loops_spatial = contains_self_loops(frame_window_edge_index_spatial)
#     has_self_loops_temporal = contains_self_loops(frame_window_edge_index_temporal)
    
#     if has_self_loops_spatial:
#         print("Graph has self-loops in spatial edges.")
#     else:
#         print("Graph does not have self-loops in spatial edges.")

#     if has_self_loops_temporal:
#         print("Graph has self-loops in temporal edges.")
#     else:
#         print("Graph does not have self-loops in temporal edges.")


In [ ]:
# from torch_geometric.utils import is_undirected

# for data in data_list:
#     frame_window_edge_index_spatial = data['frame_window', 'spatial', 'frame_window'].edge_index
#     frame_window_edge_index_temporal = data['frame_window', 'temporal', 'frame_window'].edge_index 


#     is_undirected_spatial = is_undirected(frame_window_edge_index_spatial)
#     is_undirected_temporal = is_undirected(frame_window_edge_index_temporal)
    
#     if is_undirected_spatial:
#         print("Spatial graph is undirected.")
#     else:
#         print("Spatial graph is directed.")

#     if is_undirected_temporal:
#         print("Temporal graph is undirected.")
#     else:
#         print("Temporal graph is directed.")


In [ ]:
# for data in data_list:
#     has_negative_weights = (data.edge_weight < 0).any()

# # Print the result
#     if has_negative_weights:
#         print("There are negative edge weights.")
#     else:
#         print("All edge weights are non-negative.")
    

In [ ]:
# print(data_list[3901].edge_weight)
# print(data_list[3901].edge_index)
# print(data_list[3901].num_nodes)

In [ ]:
import random
from sklearn.model_selection import train_test_split

torch.manual_seed(12345)

# Shuffle the data list
random.shuffle(data_list)

# Split the data into training and testing sets (70% train, 30% test)
train_dataset, val_dataset = train_test_split(data_list, test_size=0.3, random_state=12345)
val_dataset, test_dataset = train_test_split(val_dataset, test_size=0.5, random_state=12345)

print(f'Number of training graphs: {len(train_dataset)}')
print(f'Number of validation graphs: {len(val_dataset)}')
print(f'Number of testing graphs: {len(test_dataset)}')

In [ ]:
from torch_geometric.loader import DataLoader
from torch_geometric.loader import DataListLoader
train_loader = DataLoader(train_dataset, batch_size=128,pin_memory=True, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, pin_memory=True, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, pin_memory=True, shuffle=False)
train_batch = next(iter(train_loader))
print(train_batch)
val_batch = next(iter(val_loader))
print(val_batch)


# train_loader = DataListLoader(train_dataset, batch_size=128,pin_memory=True, shuffle=True)
# test_loader = DataListLoader(test_dataset, batch_size=128, pin_memory=True, shuffle=False)



In [ ]:
for step, data in enumerate(train_loader):
    print(f'Step {step + 1}:')
    print('=======')
    print(f'Number of graphs in the current batch: {data.num_graphs}')
    print(data)
    print()
for batch in train_loader:
    print(batch)

In [ ]:
#homogeneous classification
# labels = set()

# # Iterate through the data_list
# for data in data_list:
#     labels.add(data.y.item())

# # Convert unique_labels to a list if you need it as a list
# labels = list(labels)
# print(labels)


In [ ]:
#heterogenous classification
labels = set()

# Iterate through the data_list
for data in data_list:
    labels.add(data['frame_window'].y.item())

# Convert unique_labels to a list if you need it as a list
labels = list(labels)
print(labels)


In [ ]:
######To verify the correctness of confusion matrix########
val_labels = [data['frame_window'].y for data in val_dataset]
val_labels = torch.tensor(val_labels)
val_class_counts = torch.bincount(val_labels)
print(val_class_counts)
total_val_samples = val_class_counts.sum().item()
print(total_val_samples)


In [ ]:
# #homogeneous classification
# train_labels = [data.y for data in train_dataset]
# train_labels = torch.tensor(train_labels)
# class_counts = torch.bincount(train_labels)
# print(class_counts)
# total_samples = class_counts.sum().item()
# print(total_samples)
# class_weights = 1.0 - (class_counts / total_samples)
# print(class_weights)

In [ ]:
#heterogenous classification
train_labels = [data['frame_window'].y for data in train_dataset]
train_labels = torch.tensor(train_labels)
class_counts = torch.bincount(train_labels)
print(class_counts)
total_samples = class_counts.sum().item()
print(total_samples)
class_weights = 1.0 - (class_counts / total_samples)
print(class_weights)

In [ ]:
# #2 linear layers using GCNConv
# #homogeneous classification
# import torch
# from torch import nn
# from torch.nn import Linear
# import torch.nn.functional as F
# from torch.nn.parameter import Parameter
# import math
# import pdb
# # import time
# from torch_geometric.nn import GCNConv
# from torch.nn import BatchNorm1d
# from torch_geometric.typing import torch_scatter
# from torch_geometric.nn import global_mean_pool
# from torch_geometric.nn import global_add_pool
# from torch_geometric.nn import global_max_pool



# class RGCN(torch.nn.Module):
#     def __init__(self, in_channels, hidden_channels, out_channels):
#         super(RGCN, self).__init__()

#         # 1 by 1 conv -> 512  wang: 2048 -> 512

#         self.conv1 = GCNConv(in_channels, hidden_channels)
#         self.conv2 = GCNConv(hidden_channels, hidden_channels)
#         self.conv3 = GCNConv(hidden_channels, hidden_channels)
#         self.bn1 = nn.BatchNorm1d(hidden_channels)
#         self.bn2 = nn.BatchNorm1d(hidden_channels)
#         self.bn3 = nn.BatchNorm1d(hidden_channels)
#         self.lin1 = nn.Linear(hidden_channels*3, hidden_channels*3)
#         self.lin2 = nn.Linear(hidden_channels*3, out_channels)
# #         self.init_weight()

#     def forward(self, data):
# #         print(f'Inside model - num graphs: {data.num_graphs}, 'f'device: {data.batch.device}')
#         gc1 =F.relu(self.bn1(self.conv1(data.x, data.edge_index, data.edge_weight.sigmoid())))
#         gc2 = F.relu(self.bn2(self.conv2(gc1, data.edge_index, data.edge_weight.sigmoid())))
# #         print('shape of gc2',gc2.shape)

# #         has_nan = torch.isnan(gc2).any()

# #         if has_nan:
# #             print("The tensor contains NaN values.")
# #         else:
# #             print("The tensor does not contain NaN values.")
#         gc3 = F.relu(self.bn3(self.conv3(gc2, data.edge_index, data.edge_weight.sigmoid())))
#         out1 = global_mean_pool(gc3, data.batch)
#         out2 = global_add_pool(gc3, data.batch)
#         out3 = global_max_pool(gc3, data.batch)
#         out = torch.cat((out1, out2, out3), dim=1)
#         out = F.relu(self.lin1(out))
#         out = F.dropout(out, p=0.5, training=self.training)
#         out = self.lin2(out)
#         return out



In [ ]:
for data in train_loader:
    print(data['frame_window'].batch)

In [ ]:
#2 linear layers using GCNConv
#heterogenous classification
import torch
from torch import nn
# from torch.nn import Linear
import torch.nn.functional as F
from torch.nn.parameter import Parameter
import math
import pdb
# import time
from torch_geometric.nn import Linear,GraphConv,HeteroConv,GCNConv
from torch.nn import BatchNorm1d
from torch_geometric.typing import torch_scatter
from torch_geometric.nn import global_mean_pool
from torch_geometric.nn import global_add_pool
from torch_geometric.nn import global_max_pool



class RGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers):
        super(RGCN, self).__init__()
        self.convs1 = torch.nn.ModuleList()
        self.convs2 = torch.nn.ModuleList()
        self.convs3 = torch.nn.ModuleList()
        
        self.batch_norms1 = torch.nn.ModuleList()
        self.batch_norms2 = torch.nn.ModuleList()
        self.batch_norms3 = torch.nn.ModuleList()
        
        self.lin1 = Linear(hidden_channels*3, hidden_channels*3)
        self.lin2 = Linear(hidden_channels*3, out_channels)
        
        self.dropout = nn.Dropout(p=0.5)
        
        for _ in range(num_layers):
            conv1 = HeteroConv({('frame_window', 'spatial', 'frame_window'): GCNConv(in_channels, hidden_channels),
                                ('frame_window', 'temporal', 'frame_window'): GCNConv(in_channels, hidden_channels)}, aggr='sum')
            conv2 = HeteroConv({('frame_window', 'spatial', 'frame_window'): GCNConv(hidden_channels, hidden_channels),
                                ('frame_window', 'temporal', 'frame_window'): GCNConv(hidden_channels, hidden_channels)}, aggr='sum')
            conv3 = HeteroConv({('frame_window', 'spatial', 'frame_window'): GCNConv(hidden_channels, hidden_channels),
                                ('frame_window', 'temporal', 'frame_window'): GCNConv(hidden_channels, hidden_channels)}, aggr='sum')
            bn1 = nn.BatchNorm1d(hidden_channels)
            bn2 = nn.BatchNorm1d(hidden_channels)
            bn3 = nn.BatchNorm1d(hidden_channels)
            self.convs1.append(conv1)
            self.convs2.append(conv2)
            self.convs3.append(conv3)
            self.batch_norms1.append(bn1)
            self.batch_norms2.append(bn2)
            self.batch_norms3.append(bn3)      


    def forward(self, data):
        x_dict = data.x_dict
        edge_index_dict = data.edge_index_dict
        edge_weight_dict = data.edge_weight_dict
        batch = data['frame_window'].batch
        
        for conv1, conv2, conv3, bn1, bn2, bn3 in zip(self.convs1, self.convs2, self.convs3, 
                                                      self.batch_norms1, self.batch_norms2, self.batch_norms3):
            gc1 = conv1(x_dict, edge_index_dict, edge_weight_dict)
            gc1 = {key: bn1(x.relu()) for key, x in gc1.items()}
#             gc1 = {key: self.dropout(bn1(x.relu())) for key, x in gc1.items()} 
            gc2 = conv2(gc1, edge_index_dict, edge_weight_dict)
            gc2 = {key: bn2(x.relu()) for key, x in gc2.items()}
#             gc2 = {key: self.dropout(bn2(x.relu())) for key, x in gc2.items()}
            gc3 = conv3(gc2, edge_index_dict, edge_weight_dict)
            gc3 = {key: bn3(x.relu()) for key, x in gc3.items()}
#             gc3 = {key: self.dropout(bn3(x.relu())) for key, x in gc3.items()}

        out1 = global_mean_pool(gc3['frame_window'], batch)
        out2 = global_add_pool(gc3['frame_window'], batch)
        out3 = global_max_pool(gc3['frame_window'], batch) 
        out = torch.cat((out1, out2, out3), dim=1)
        out = F.relu(self.lin1(out))
        out = F.dropout(out, p=0.5, training=self.training)
        out = self.lin2(out)
        return out



In [ ]:
# #2 linear layers using SplineConv
# import torch
# from torch import nn
# from torch.nn import Linear
# import torch.nn.functional as F
# from torch.nn.parameter import Parameter
# import math
# import pdb
# # import time
# from torch_geometric.nn import GCNConv
# from torch.nn import BatchNorm1d
# from torch_geometric.nn import SplineConv
# import torch_spline_conv
# from torch_geometric.typing import torch_scatter
# from torch_geometric.nn import global_mean_pool
# from torch_geometric.nn import global_add_pool
# from torch_geometric.nn import global_max_pool



# class RGCN(torch.nn.Module):
#     def __init__(self, in_channels, hidden_channels, out_channels):
#         super(RGCN, self).__init__()

#         # 1 by 1 conv -> 512  wang: 2048 -> 512

#         self.conv1 = SplineConv(in_channels, hidden_channels,dim=2, kernel_size=5, aggr='add')
#         self.conv2 = SplineConv(hidden_channels, hidden_channels, dim=2, kernel_size=5, aggr='add')
#         self.conv3 = SplineConv(hidden_channels, hidden_channels, dim=2, kernel_size=5, aggr='add')
#         self.conv4 = SplineConv(hidden_channels, hidden_channels, dim=2, kernel_size=5, aggr='add')
#         self.conv5 = SplineConv(hidden_channels, hidden_channels, dim=2, kernel_size=5, aggr='add')
#         self.conv6 = SplineConv(hidden_channels, hidden_channels,dim=2, kernel_size=5, aggr='add')
# #         self.bn1 = BatchNorm1d(hidden_channels)
# #         self.bn2 = BatchNorm1d(hidden_channels)
# #         self.bn3 = BatchNorm1d(hidden_channels)
#         self.lin1 = Linear(hidden_channels*3, hidden_channels*3)
#         self.lin2 = Linear(hidden_channels*3, out_channels)
# #         self.init_weight()

#     def forward(self, data):
# #         print(f'Inside model - num graphs: {data.num_graphs}, 'f'device: {data.batch.device}')
#         gc1 = F.elu(self.conv1(data.x, data.edge_index, data.edge_weight.sigmoid()))
#         gc2 = F.elu(self.conv2(data.x, data.edge_index, data.edge_weight.sigmoid()))
#         gc3 = F.elu(self.conv3(data.x, data.edge_index, data.edge_weight.sigmoid()))
#         gc4 = F.elu(self.conv4(data.x, data.edge_index, data.edge_weight.sigmoid()))
#         gc5 = F.elu(self.conv5(data.x, data.edge_index, data.edge_weight.sigmoid()))
#         gc6 = F.elu(self.conv6(data.x, data.edge_index, data.edge_weight.sigmoid()))
#         out1 = global_mean_pool(gc6, data.batch)
#         out2 = global_add_pool(gc6, data.batch)
#         out3 = global_max_pool(gc6, data.batch)
#         out = torch.cat((out1, out2, out3), dim=1)
#         out = F.relu(self.lin1(out))
#         out = F.dropout(out, p=0.5, training=self.training)
#         out = self.lin2(out)
#         return out

In [ ]:

in_channels = data_list[0].num_features['frame_window'] 
hidden_channels = 64
out_channels = len(labels)
print(in_channels)
print(out_channels)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RGCN(in_channels, hidden_channels, out_channels,2)
model = model.to(device)
print(model)

In [ ]:
# for batch_id, data in enumerate(train_loader):
#     for key, x in data.x_dict.items():
#         print(f"{key} input features shape: {list(x.shape)}")

In [ ]:
for name, param in model.named_parameters():
    print(f"Parameter: {name}, Requires Grad: {param.requires_grad}")

In [ ]:
# #for homogenous graphs
# import datetime
# import torch_scatter
# from sklearn.metrics import accuracy_score
# from sklearn import metrics
# from sklearn.metrics import confusion_matrix
# import matplotlib.pyplot as plt
# import seaborn as sns

# lr = 0.00001
# optimizer = torch.optim.Adam(model.parameters(), lr=lr)
# # optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
# class_weights = class_weights.to(device)
# loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
# all_preds = []
# all_labels = []

# def train(epochs):
#     global all_preds, all_labels
#     for epoch in range(1,epochs+1):
#         all_preds = []
#         all_labels = []
#         loss_train = 0.0
#         correct_train = 0
#         total =0
#         total_batch =0
#         model.train()
#         for batch_id, data in enumerate(tqdm(train_loader)):
            
#             data = data.to(device)
            
#             out = model(data)
#             with torch.no_grad():
#                 pred = out.argmax(dim=1)
                
#             loss = loss_fn(out, data.y) 
#             optimizer.zero_grad()
#             loss.backward()        
#             optimizer.step() 
#             loss_train  += loss.item()  
#             correct_train += int((pred == data.y).sum())
#             total += len(data)
#             total_batch += 1
#         model.eval()    
#         total_val, total_val_batch, loss_val, correct_val, all_labels_epoch,all_preds_epoch = validate_model()
#         all_labels.extend(all_labels_epoch)
#         all_preds.extend(all_preds_epoch) 
#         print('{} Epoch-{}, train-loss {:.4f}, train-acc {:.4f} || val-loss {:.4f}, val-acc {:.4f} || lr - {}'.format(
#             datetime.datetime.now(), epoch, loss_train/total_batch, correct_train/total, loss_val/total_val_batch,
#             correct_val/total_val, lr))
#     cm = confusion_matrix(all_labels, all_preds)
#     overall_accuracy = accuracy_score(all_labels, all_preds)
#     print(overall_accuracy)
#     class_accuracy = cm.diagonal() / cm.sum(axis=1)
#     for class_idx, acc in enumerate(class_accuracy):
#         print(f'Class {class_idx} Accuracy: {acc}')
#     # Plot confusion matrix
#     plt.figure(figsize=(10, 8))
#     sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', xticklabels=range(len(class_accuracy)), yticklabels=range(len(class_accuracy)))
#     plt.xlabel('Predicted')
#     plt.ylabel('Actual')
#     plt.title('Confusion Matrix')
#     plt.show()
        
# def validate_model():
#     all_preds_epoch = []
#     all_labels_epoch = []
#     model.eval()

#     correct = 0
#     total = 0
#     total_batch = 0
#     loss_val =0.0

    
#     with torch.no_grad():
#         for batch_id, data in enumerate(tqdm(test_loader)):
#             data = data.to(device)
#             out = model(data)
#             pred = out.argmax(dim=1)
#             total += len(data)
#             total_batch += 1
#             correct += int((pred == data.y).sum())
#             loss = loss_fn(out, data.y)
#             loss_val += loss.item()
#             all_preds_epoch.extend(pred.cpu().numpy())
#             all_labels_epoch.extend(data.y.cpu().numpy()) 
#     return total, total_batch, loss_val, correct,all_labels_epoch,all_preds_epoch  

In [ ]:
#for heterogenous graphs
import datetime
# import torch_scatter
from sklearn.metrics import accuracy_score
from sklearn import metrics
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
# import seaborn as sns
from torch import optim

lr = 0.00001
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.7, patience=2,
#                                                               min_lr=1e-7)
class_weights = class_weights.to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
all_preds = []
all_labels = []

def train(epochs):
    early_stopper = EarlyStopper()
    global all_preds, all_labels
    train_losses = []
    val_losses = []
    for epoch in range(1,epochs+1):
        all_preds = []
        all_labels = []
        loss_train = 0.0
        correct_train = 0
        total =0
        total_batch =0
        model.train()
        for batch_id, data in enumerate(tqdm(train_loader)):
            
            data = data.to(device)
            
            out = model(data)
            with torch.no_grad():
                pred = out.argmax(dim=1)
                
            loss = loss_fn(out, data['frame_window'].y) 
            optimizer.zero_grad()
            loss.backward()        
            optimizer.step() 
            loss_train  += loss.item()  
            correct_train += int((pred == data['frame_window'].y).sum())
            total += len(data)
            total_batch += 1
        model.eval()    
        total_val, total_val_batch, loss_val, correct_val, all_labels_epoch,all_preds_epoch = validate_model()
#         scheduler.step(loss_val/total_val)
#         current_lr = get_lr()
        all_labels.extend(all_labels_epoch)
        all_preds.extend(all_preds_epoch) 
        train_losses.append(loss_train / total_batch)
        val_losses.append(loss_val / total_val_batch)
        print('{} Epoch-{}, train-loss {:.4f}, train-acc {:.4f} || val-loss {:.4f}, val-acc {:.4f} || lr - {}'.format(
            datetime.datetime.now(), epoch, loss_train/total_batch, correct_train/total, loss_val/total_val_batch,
            correct_val/total_val, lr))
        # Check to see if there is early stopping
        if early_stopper.early_stop(loss_val):
            break
            
    plt.plot(range(1, epoch + 1), train_losses, label='Train Loss')
    plt.plot(range(1, epoch + 1), val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()
    
    cm = confusion_matrix(all_labels, all_preds)
    cmn = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    overall_accuracy = accuracy_score(all_labels, all_preds)
    print(overall_accuracy)
    class_accuracy = cm.diagonal() / cm.sum(axis=1)
    for class_idx, acc in enumerate(class_accuracy):
        print(f'Class {class_idx} Accuracy: {acc}')
    # Plot confusion matrix
    plt.figure(figsize=(15, 6))
    plt.subplot(1, 2, 1)
    sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', xticklabels=range(len(class_accuracy)), yticklabels=range(len(class_accuracy)))
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()
    
    plt.subplot(1, 2, 2)
    sns.heatmap(cmn, annot=True, fmt='.2%', cmap='Blues', xticklabels=range(len(class_accuracy)), yticklabels=range(len(class_accuracy)))
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Normalized Confusion Matrix')
    plt.show()
        
def validate_model():
    all_preds_epoch = []
    all_labels_epoch = []
    model.eval()

    correct = 0
    total = 0
    total_batch = 0
    loss_val =0.0

    
    with torch.no_grad():
        for batch_id, data in enumerate(tqdm(val_loader)):
            data = data.to(device)
            out = model(data)
            pred = out.argmax(dim=1)
            total += len(data)
            total_batch += 1
            correct += int((pred == data['frame_window'].y).sum())
            loss = loss_fn(out, data['frame_window'].y)
            loss_val += loss.item()
            all_preds_epoch.extend(pred.cpu().numpy())
            all_labels_epoch.extend(data['frame_window'].y.cpu().numpy()) 
    return total, total_batch, loss_val, correct,all_labels_epoch,all_preds_epoch  

class EarlyStopper:
    """
    The EarlyStopper class for training
    """

    def __init__(self, patience=30, min_delta=0.015):
        """
        Initialization

        :param patience: Number of epochs to wait
        :param min_delta: The minimum change that is required
        """

        # Init
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = np.inf

    def early_stop(self, validation_loss):
        """
        Early stop instance

        :param validation_loss: The loss to monitor
        :return:
        """

        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                sys.stdout.write("EarlyStopping the training process due to non changing validation loss\n")
                return True
        return False
    
def get_lr():

    """
    Get the current learning rate for the model
    :return: The learning rate for the current epoch after the update
    """

    for param_group in optimizer.param_groups:
        return param_group["lr"]


In [ ]:
train(500)
#save trained model
torch.save(model.state_dict(), 'rgcn_model.pt')

In [ ]:
with open("data.yml", "r") as file_handle:
    yaml_file_params = yaml.load(file_handle, Loader=yaml.Loader)

# Assembly operation selected
assembly = 'L10'

# Identify the assembly operation data
assembly_operation = yaml_file_params[assembly]

# Testing data location
testing_videos_dir = os.path.join(os.path.dirname(os.getcwd()), *assembly_operation["data_dir"]["testing"])
testing_video_names = [os.path.join(testing_videos_dir, cycle) for cycle in os.listdir(testing_videos_dir) if
                        cycle.split(".")[-1] == "mp4"]
testing_annotation_names = ["human_" + cycle.split("/")[-1][:-4] + ".csv" for cycle in testing_video_names]

# Assert that all the testing data is labelled
assert len(testing_video_names) == len(testing_annotation_names), "The labelling and the data for testing " \
                                                                    "does not match"
# Get the full path of the testing data
testing_videos_fullpath = [os.path.join(testing_videos_dir, cycle_name) for cycle_name in
                            testing_video_names]

testing_annotations_fullpath = [os.path.join(testing_videos_dir, annotation_name) for annotation_name in
                                 testing_annotation_names]


RESULTS_DIR = "/data1/GCNN/RPN_Test_Results"
window_size = 30
overlap = 15


In [ ]:
# Make a final assertion to ensure they match
start_time = time.time()
testing_data_list = []
all_tasks = pd.DataFrame()
for video_name, annotation_name in zip(testing_videos_fullpath, testing_annotations_fullpath):
    aug_video_name = video_name.split(os.path.sep)[-1].split(".")[0]
    aug_annotation_name = ("_".join(annotation_name.split(os.path.sep)[-1].split("_")[1:])).split(".")[0]
    assert aug_video_name == aug_annotation_name, "The annotations and the cycle do not match. Please check " \
                                                  "testing directory"
# Load into DataFrame
y_list = []
# Initialize frame_data_mapping dictionary
for annotation_path in testing_annotations_fullpath:
    y_list.append(pd.read_csv(annotation_path, header=0, names=["task", "startTime", "endTime"]))
# Process the DataFrame
# y = []
video_counter = 0
# all_frame_window_tasks = []  # Initialize the frame_window_data list
total_frame_window_count = 0  # Initialize the counter for total frame windows

for video, annotation in zip(testing_videos_fullpath, y_list):
    frame_data_mapping = {}
    y = []
    combined_y = None
#     if 'WIN_20220128_14_20_20_Pro_Jan28_cycle1' not in video:
#         continue 

    processed_y = modify_annotation_data(input_video_path=video, annotation_list=annotation)
    y.append(processed_y)
    # Stack the y together
    imp_cols = ["video", "task", "startTime", "endTime", "start_frame", "end_frame"]
    for index, df_y in enumerate(y):
        # Set the video ID
        df_y["video"] = video_counter

        if index == 0:
            combined_y = df_y[imp_cols]
            index += 1
        else:
            combined_y = pd.concat([combined_y, df_y[imp_cols]], axis=0, ignore_index=True)
    all_tasks = pd.concat([all_tasks, combined_y])
    sys.stdout.write(f"Total number of task instances: {combined_y['task'].count()}")
    # print("\n")
    video_counter += 1
    combined_y["start_frame"] = combined_y["start_frame"].astype(int)
    combined_y["end_frame"] = combined_y["end_frame"].astype(int)
    print("video:", video)
#     print(combined_y)

    for index, task in combined_y.iterrows():
        seq = list(range(int(task['start_frame']), int(task['end_frame']) + 1))
        frame_window_list = GetShiftingWindows(seq, window_size, overlap)

        for frame_window in frame_window_list:
            frame_metadata_list = []  # To store the frame metadata for each frame in the frame window
            labels, tasks = load_labels(combined_y, frame_window)
            adjusted_frame_window = []  # List to store adjusted frame window

            for frames in frame_window:
                frame_path = os.path.join(RESULTS_DIR, video.split('/')[-1][:-4], str(frames) + '.pickle')
                if os.path.exists(frame_path):
                    with open(frame_path, 'rb') as handle:
                        frame_metadata = pickle.load(handle)
                        frame_metadata['features'] = frame_metadata['features'].cpu()
                        frame_metadata_list.append(frame_metadata) #store corresponding metadata for each frame window
                        adjusted_frame_window.append(frames)  # Add the frame to the adjusted frame window
            if len(adjusted_frame_window) == 0:
                continue
            frame_data_mapping[tuple(adjusted_frame_window)] = {"metadata": frame_metadata_list, "task": tasks}

            # Associate frame_window with frame_metadata list and task labels

    # Number of frame_windows
    print("Number of frame_windows per video:", len(frame_data_mapping))
    torch.cuda.empty_cache()
    
    for index,(frame_window, frame_window_data) in enumerate(frame_data_mapping.items()):# inside all frame windows in a specific video
#         print(frame_window)
        total_object_count =0
        #total_object_count_classes =0
        original_frame_window_features = [] # Initialize the list to store original features for each frame window in the current video
        frame_window_features = [] # Initialize the list to store modified features for each frame window in the current video
        frame_window_classes =[] # Initialize the list to store classes for each frame window
        frame_window_boxes =[] # Initialize the list to store boxes for each frame window
        frame_window_tasks =[]
        frame_window_tasks.append(frame_window_data['task'])
#         print(frame_window_tasks)
        total_frame_window_count += 1
        frame_window_metadata_list = frame_window_data["metadata"] #all frame metadata for all the frames within a specific frame window
        # inside all frames within a specific frame window within a specific video:
        for frame_window_idx, frame_window_metadata in enumerate(frame_window_metadata_list):
            frame_features = frame_window_metadata["features"] #features of all the frames within a specific frame window
            frame_boxes = frame_window_metadata["boxes"] #bounding box coordinates of all the frames within a specific frame window
            frame_classes = frame_window_metadata["class"] #classes of all the frames within a specific frame window
            
            total_object_count+=len(frame_boxes)
            frame_window_classes.append(frame_classes)
#         print(frame_window_classes)
            frame_window_boxes.append(frame_boxes)
            original_frame_window_features.append(frame_features)
     
            for object_idx, object_feature in enumerate(frame_features):# inside all objects within a specific frame
                max_pool = nn.MaxPool2d(kernel_size=(7, 7))
                pooled_features_per_object = max_pool(object_feature)
                averaged_features_per_object = torch.mean(pooled_features_per_object, dim=(-2, -1))
                frame_window_features.append(averaged_features_per_object)
                
        frame_window_features = torch.stack(frame_window_features, dim=0)
        row_sums = frame_window_features.sum(dim=1, keepdim=True)
        frame_window_features = frame_window_features / row_sums
        labels = encode_onehot(all_tasks,frame_window_tasks)

        labels = torch.LongTensor(np.where(labels)[1])

        adj_spatial, adj_temporal = construct_iou_adjacency_matrix(frame_window_classes,frame_window_boxes,
                                             total_object_count)
        edge_index_spatial = adj_spatial.nonzero().t()
        edge_index_spatial = edge_index_spatial.to(torch.long)      
        edge_weight_spatial = adj_spatial[edge_index_spatial[0], edge_index_spatial[1]]
        edge_weight_spatial = edge_weight_spatial.to(torch.float)

        edge_index_temporal = adj_temporal.nonzero().t()
        edge_index_temporal = edge_index_temporal.to(torch.long)
        edge_weight_temporal = adj_temporal[edge_index_temporal[0], edge_index_temporal[1]]
        edge_weight_temporal = edge_weight_temporal.to(torch.float)
        data = HeteroData(frame_window={'x': frame_window_features})
        data['frame_window'].y = labels
        data['frame_window','spatial','frame_window'].edge_index = edge_index_spatial
        data['frame_window','spatial','frame_window'].edge_weight = edge_weight_spatial
        data['frame_window','temporal','frame_window'].edge_index = edge_index_temporal        
        data['frame_window','temporal','frame_window'].edge_weight = edge_weight_temporal
        testing_data_list.append(data)
num_data_objects = len(testing_data_list)
print(f"Number of Data objects in the list: {num_data_objects}")

print("--- %s seconds ---" % (time.time() - start_time))
# print(all_tasks)

In [ ]:
from torch_geometric.loader import DataLoader
from torch_geometric.loader import DataListLoader
test_loader = DataLoader(data_list, batch_size=1, pin_memory=True, shuffle=False)
test_batch = next(iter(testing_data_list))
print(test_batch)


In [ ]:
######To verify the correctness of confusion matrix########
test_labels = [data['frame_window'].y for data in test_dataset]
test_labels = torch.tensor(test_labels)
test_class_counts = torch.bincount(test_labels)
print(test_class_counts)
total_test_samples = test_class_counts.sum().item()
print(total_test_samples)


In [ ]:
import datetime
# import torch_scatter
from sklearn.metrics import accuracy_score
from sklearn import metrics
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
# MODEL_PATH = '/data1/GCNN/rgcn_model.pt'
# model.load_state_dict(torch.load(MODEL_PATH))

model.eval()

all_preds = []
all_labels = []
correct = 0
total = 0
total_batch = 0
loss_test = 0.0
with torch.no_grad():
     for batch_id, data in enumerate(tqdm(val_loader)):
            data = data.to(device)
            out = model(data)
            pred = out.argmax(dim=1)
            total += len(data)
            total_batch += 1
            correct += int((pred == data['frame_window'].y).sum())
#             loss = loss_fn(out, data['frame_window'].y)
#             loss_test += loss.item()
            all_preds.append(pred.cpu().numpy())
            all_labels.append(data['frame_window'].y.cpu().numpy())
            
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)           
overall_accuracy = accuracy_score(all_labels, all_preds)
print(overall_accuracy)
cm = confusion_matrix(all_labels, all_preds)
cmn = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
class_accuracy = cm.diagonal() / cm.sum(axis=1)
for class_idx, acc in enumerate(class_accuracy):
    print(f'Class {class_idx} Accuracy: {acc}')
# Plot confusion matrix
plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)
sns.heatmap(cmn, annot=True, fmt='.2%', cmap='Blues', xticklabels=range(len(class_accuracy)), yticklabels=range(len(class_accuracy)))
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Normalized Confusion Matrix')
plt.show() 

plt.subplot(1, 2, 2)
sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', xticklabels=range(len(class_accuracy)), yticklabels=range(len(class_accuracy)))
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
# Create good plots
import matplotlib
font = {'family' : 'sans-serif',
        'size'   : 12}

matplotlib.rc('font', **font)
labels = ["S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S12"]
fig = plt.figure(figsize=(10, 8))
axs = fig.add_axes([0, 0, 1, 1])
sns.heatmap(cmn, annot=True, fmt='.2%', cmap="Blues", xticklabels=labels, yticklabels=labels, ax=axs)
fig.savefig("output.png", bbox_inches="tight")

In [ ]:
# F1 Scores for each class
from sklearn.metrics import f1_score, classification_report
f1_score(all_labels, all_preds, average=None)

In [ ]:
print(classification_report(all_labels, all_preds))

In [ ]:
f1_score(all_labels, all_preds, average="weighted")

In [ ]:
from collections import Counter
Counter(all_labels)

In [ ]:
print(class)